In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, bi, dictionary, maintain, observe

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
print(cfg["code"], env, "bi views:", list(bi.wanted(cfg)))

In [0]:
start = time.time()
print("new-column watch (ops.rescued_keys):", observe.rescued_keys(spark, cfg), "rows")   # 5.12, before the views
built = bi.build(spark, cfg)
for name, rows in built.items():
    print(f"{name:24} {rows:>6} rows")
print("TOTAL", built, f"in {time.time() - start:.0f}s")
words = dictionary.apply(spark, cfg)                     # data dictionary: table and column descriptions (after Gold)
print(f"dictionary: {words['tables']} table and {words['columns']} column descriptions set, {words['unchanged']} unchanged")
for k in ("missing", "undocumented", "failed"):
    if words[k]:
        print(f"dictionary {k} ({len(words[k])}):", words[k][:20])
packed, skipped = maintain.compact(spark, cfg)           # small-file tables (config/performance.yml compact)
print("compacted:", ", ".join(f"{t} {a} -> {b} files" for t, a, b in packed) or "nothing needed")
if skipped:
    print("compaction skipped:", skipped)
common.log_run(spark, cfg, "bi", "success", sum(built.values()),
               f"{built} dictionary set {words['tables']}+{words['columns']} failed {len(words['failed'])}")